# DRISHTI — full pipeline on a Colab / Kaggle GPU (run EVERYTHING here)

You have the light spine locally but **not** the heavy dependencies (COLMAP, Open3D, Torch, …), so a
local run fails loudly at `s2_poses`. This notebook installs the full stack on a free **T4** and runs
the **entire 11-stage pipeline end-to-end on the GPU** — `s0_ingest → … → report` — then hands you the
whole bundle (model + deliverables + accuracy report + **logs**) to download.

> Sibling notebook `drishti_cloud_t4.ipynb` does the *tier split* (heavy stages on the T4, then
> `resume` the light stages on your machine). **This** notebook does the opposite: **all** stages here,
> nothing left for the ground. Both write the same content-hashed bundle.

**Honest by construction:** it installs the real package, runs the real `drishti` CLI, and prints the
real `doctor` / `inspect` / `verify` output. No stubs, no fabricated numbers — a missing dependency,
GPU, or input **fails loudly** and says exactly what is missing.

### Run order
1. **Runtime → Change runtime type → T4 GPU** (Colab) / enable the GPU accelerator (Kaggle).
2. Set `REPO_URL` in §1.
3. Run §0–§4 (GPU → code → install → doctor).
4. In §5 pick **one** import option (generate here / your own video / resume a partial bundle).
5. Run §6 (full pipeline), then §7–§9 (inspect → logs → download).

## 0 · Confirm the GPU

If no Tesla GPU is listed, stop and switch the runtime to GPU — the neural (S3/S4) and dense (S7) stages
need CUDA.

In [ ]:
!nvidia-smi

## 1 · Get the code

Point these at your DRISHTI repository. On Kaggle you can instead attach the repo as a dataset and set
`REPO_DIR` to its path (e.g. `/kaggle/input/drishti`), then skip the clone.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/drishti.git"   # <-- set me
REPO_REF = "main"                                         # branch, tag, or commit
REPO_DIR = Path("/content/drishti") if Path("/content").exists() else Path("/kaggle/working/drishti")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"{REPO_DIR} already present — skipping clone")

os.chdir(REPO_DIR)

# Defaults so §6 never NameErrors even if you run only one import cell in §5.
MISSION = ""            # a dataset descriptor .yaml  -> fresh full run
RESUME_BUNDLE = None    # a partial bundle directory  -> resume on the GPU
print("cwd:", Path.cwd())

## 2 · Install DRISHTI (auto-handles Colab's Python 3.13)

This cell is self-healing and installs **each group in isolation**, so one un-buildable wheel can never
abort the whole install (the failure that leaves you with `drishti: command not found`). DRISHTI core is
installed **first**, so the CLI always lands.

**The Python 3.13 catch (why this cell may build a second interpreter).** `s7_dense` (TSDF) and
`s8_mesh` (Poisson) need **Open3D**, and Open3D publishes wheels only up to **cp312**. Current Colab /
Kaggle run **Python 3.13**, for which no Open3D wheel exists. So if the kernel is > 3.12, the cell uses
[`uv`](https://github.com/astral-sh/uv) to build an isolated **Python 3.12** environment, installs
everything there, and prepends it to `PATH` — every later cell's `!drishti` / `!python` then transparently
uses it. (On a ≤ 3.12 kernel it just installs in place and keeps the hosted CUDA PyTorch.) This is the
honest, working path to a **full** run — Open3D and all — on today's runtimes.

Beyond the `pyproject` extras it also installs two packages the extras don't list but a full run needs:

- **`transformers`** — S3 (RT-DETR) and S4 (Depth-Anything-V2) load their models from HuggingFace
  `transformers`; the `depth` extra ships only the `torch/openvino/onnx` *runtimes*, not `transformers`.
- **`laspy`** — S10 writes the **LAS** point cloud (a MUST deliverable); it isn't in any extra either.

The cell ends with a capability probe (`OK` / `MISSING` per module) — the same imports §4 `doctor` gates
each stage on. If the 3.12 build ever can't be created in your session (e.g. `uv` can't fetch a Python),
tell me and I'll switch this cell to the `condacolab` route instead.

In [ ]:
# DRISHTI's dense/mesh stages (s7_dense TSDF, s8_mesh Poisson) need Open3D, which publishes wheels only
# through cp312. Colab/Kaggle now ship Python 3.13, for which NO open3d wheel exists — so a single
# `pip install .[recon]` aborts and takes the whole install (and the `drishti` CLI) down with it.
# Fix: if the kernel is newer than 3.12, build a uv-managed Python 3.12 env and install EVERYTHING there,
# then put it first on PATH so every later cell (!drishti, !python, subprocess) transparently uses it.
# Nothing is stubbed — whatever a stage needs is really installed, and §4 `doctor` reports the truth.
import os, sys, subprocess
from pathlib import Path

def sh(cmd):
    print("  $", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str)).returncode

pyver = sys.version_info[:2]
USE_VENV = pyver > (3, 12)   # open3d has no wheel past cp312
print(f"kernel Python {pyver[0]}.{pyver[1]}  ->  "
      f"{'building an isolated Python 3.12 env (open3d has no wheel here)' if USE_VENV else 'installing in place'}\n")

if not USE_VENV:
    PY = sys.executable
else:
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    VENV = (Path("/content") if Path("/content").exists() else Path("/kaggle/working")) / "venv312"
    sh(["uv", "python", "install", "3.12"])
    if not (VENV / "bin" / "python").exists():
        sh(["uv", "venv", "--seed", "--python", "3.12", str(VENV)])   # --seed => venv gets pip, so later !pip works too
    bindir = VENV / "bin"
    os.environ["VIRTUAL_ENV"] = str(VENV)
    os.environ["PATH"] = f"{bindir}{os.pathsep}{os.environ['PATH']}"   # !drishti / !python / subprocess -> this venv
    PY = str(bindir / "python")
    print("\nusing", PY, "\n")

def pipi(*pkgs, required=False):
    """Install into the target interpreter. Isolated: one failing native wheel can't abort the rest."""
    cmd = (["uv", "pip", "install", "--python", PY] if USE_VENV else [PY, "-m", "pip", "install", "-q"]) + list(pkgs)
    if sh(cmd) != 0:
        if required:
            raise RuntimeError(f"REQUIRED install failed: {' '.join(pkgs)} — cannot continue.")
        print(f"  ! optional install failed: {' '.join(pkgs)} — §4 `doctor` will show what's blocked\n")

# A fresh 3.12 venv has no torch; the hosted (>=3.13) kernel already ships the right CUDA torch, so keep that.
if USE_VENV:
    pipi("torch")   # PyPI Linux torch is the CUDA build; it uses the host's NVIDIA driver
subprocess.run([PY, "-c", "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])

# 1) DRISHTI core — REQUIRED, first, so the `drishti` CLI always lands even if a heavy extra can't build.
pipi("-e", ".", required=True)

# 2) Heavy capability groups from pyproject.toml, each isolated (recon = open3d/trimesh, poses = pycolmap, ...).
for grp in ("video", "geo", "recon", "poses", "spine", "telem", "depth", "server"):
    pipi("-e", f".[{grp}]")

# 3) The two packaging gaps the extras don't list but a full run needs, + Pillow (real-dataset EXIF path):
#    transformers -> S3 (RT-DETR) & S4 (Depth-Anything-V2) models;  laspy -> S10 LAS deliverable.
pipi("-U", "transformers>=4.45", "timm", "safetensors")
pipi("laspy[lazrs]>=2.5")
pipi("pillow")

# 4) Honest capability probe — the same imports §4 `doctor` gates each stage on.
probe = (
    "import importlib.util as u\n"
    "mods=['av','cv2','pyproj','rasterio','shapely','open3d','trimesh','pycolmap','gtsam','torch','transformers','laspy']\n"
    "print('\\ninstalled capabilities:')\n"
    "for m in mods:\n"
    "    print('  %-13s %s' % (m, 'OK' if u.find_spec(m) else 'MISSING'))\n"
)
subprocess.run([PY, "-c", probe])

## 3 · (optional) Blender — so the FBX deliverable is real, not skipped

**FBX** is a required output. S10 produces it out-of-process with **headless Blender ≥ 4.0** (it calls
`bpy.ops.wm.ply_import`, which exists only in Blender 4.x — the distro's `apt` Blender is too old). This
cell fetches a portable Blender 4.2 LTS and puts it on `PATH`.

Skip this cell if you don't need FBX: it is then recorded as *skipped* and `s10_export` marked
**degraded** (not failed) — every other format still exports.

In [ ]:
import os, shutil, subprocess, urllib.request
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path("/kaggle/working")

def _fetch(url, dest):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "drishti"})
        with urllib.request.urlopen(req, timeout=90) as r:
            if getattr(r, "status", 200) >= 400:
                return False
            with open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
        return dest.stat().st_size > 1_000_000
    except Exception as e:
        print("   ", url.rsplit("/", 1)[-1], "->", e)
        return False

if shutil.which("blender"):
    print("blender already on PATH:", shutil.which("blender"))
else:
    tarxz = BASE / "blender.tar.xz"
    got = None
    for v in ["4.2.5", "4.2.4", "4.2.3", "4.2.1", "4.2.0"]:
        print("trying Blender", v)
        if _fetch(f"https://download.blender.org/release/Blender4.2/blender-{v}-linux-x64.tar.xz", tarxz):
            got = v
            break
    if not got:
        print("\n! Blender not fetched — FBX will be skipped (s10_export degraded). This is non-fatal.")
    else:
        dest = BASE / "blender"
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir()
        subprocess.run(["tar", "-xf", str(tarxz), "-C", str(dest), "--strip-components=1"], check=True)
        link = Path("/usr/local/bin/blender")
        if link.exists() or link.is_symlink():
            link.unlink()
        link.symlink_to(dest / "blender")
        print(subprocess.run(["blender", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 4 · Doctor — the honest gate (spend GPU minutes only if this is green)

Expect **cuda: yes** and every stage **ready** (all extras + the two gap packages installed; the FBX
tool present if you ran §3). Anything still `blocked` here will fail loudly when reached — fix it first.

In [ ]:
!drishti doctor

## 5 · Provide inputs (IMPORT) — run **ONE** of the next three cells

| Option | Use when | What it sets |
|--------|----------|--------------|
| **D · generate here** | first real run / you have no data of your own | `MISSION` (fresh full run) |
| **A/B · your own mission** | you have a video + telemetry + descriptor | `MISSION` (fresh full run) |
| **C · resume a partial bundle** | you already ran S0/S1 locally and want to continue | `RESUME_BUNDLE` |

A DRISHTI input is a `mission.yaml` **descriptor** plus the files it points at (video + GPS track, and
any optional IMU / baro / intrinsics / RTK). See `docs/GUIDE.md` for the descriptor contract.

In [ ]:
# == IMPORT · Option D — GENERATE a realistic dataset on this GPU (no upload) ==================
# Assembles the real MP4+SRT contract from open, GPS-tagged aerial imagery (or a ground-truth
# synthetic city). Only playback timing is synthesized; pixels + GPS are the source's own.
DATASET = "aukerman"   # aukerman(real buildings ~543MB) | brighton_beach(fast ~62MB) | caliterra | lewis | synthetic

if DATASET == "synthetic":
    !python scripts/make_sample_dataset.py --synthetic
    _name = "synthetic_city"
else:
    !python scripts/make_sample_dataset.py --dataset {DATASET}
    _name = DATASET

import os
from pathlib import Path
MISSION = str(REPO_DIR / "configs" / "datasets" / f"{_name}.yaml")
RESUME_BUNDLE = None
os.environ["DRISHTI_MISSION"] = MISSION
assert Path(MISSION).is_file(), MISSION
print("\nMISSION =", MISSION, "\n")
print(Path(MISSION).read_text())

In [ ]:
# == IMPORT · Options A/B — bring YOUR OWN mission (fresh full run) =============================
# Uncomment ONE. Point at the descriptor AND make sure the files it references are present.
from pathlib import Path

# A) Google Drive (best for a large video):
# from google.colab import drive; drive.mount("/content/drive")
# MISSION = "/content/drive/MyDrive/drishti/my_mission.yaml"

# B) Direct upload of small files (descriptor + video + telemetry together):
# from google.colab import files; up = files.upload()
# MISSION = next(k for k in up if k.endswith((".yaml", ".yml")))

RESUME_BUNDLE = None
assert MISSION and Path(MISSION).is_file(), (
    f"MISSION not set/found ({MISSION!r}). Uncomment A or B above, or use Option D.")
print("mission:", MISSION)

In [ ]:
# == IMPORT · Option C — RESUME a partial bundle on the GPU ====================================
# You ran S0/S1 locally and S2 failed for lack of GPU deps. Continue that exact bundle here.
#
# IMPORTANT: the runner re-checks S0's inputs, so the descriptor's video + telemetry must exist at
# their recorded relative paths. Either regenerate them (if they came from make_sample_dataset — same
# paths) or upload/mount your originals BEFORE resuming. If the files match what the local run hashed,
# S0/S1 are skipped (fresh); otherwise they recompute — either way it runs through to completion.
RUN_OPTION_C = False   # <-- set True to use this cell

if RUN_OPTION_C:
    import zipfile
    from pathlib import Path
    # (a) make the ORIGINAL inputs available at their recorded paths, e.g. regenerate a sample set:
    # !python scripts/make_sample_dataset.py --dataset brighton_beach
    # (b) upload the partial bundle you zipped locally (PowerShell:  Compress-Archive runs\<id> <id>.zip)
    RUNS_DIR = REPO_DIR / "runs"
    RUNS_DIR.mkdir(exist_ok=True)
    from google.colab import files
    up = files.upload()
    zname = next(k for k in up if k.endswith(".zip"))
    with zipfile.ZipFile(zname) as z:
        z.extractall(RUNS_DIR)
    cand = sorted(RUNS_DIR.rglob("manifest.json"), key=lambda p: p.stat().st_mtime)
    assert cand, "no manifest.json found after unzip — zip the run_id FOLDER (not the runs/ parent)."
    RESUME_BUNDLE = str(cand[-1].parent)
    MISSION = ""
    print("will RESUME:", RESUME_BUNDLE)
else:
    print("Option C disabled (RUN_OPTION_C = False). Using Option D/A/B's MISSION.")

## 6 · Run the FULL pipeline (all 11 stages) — run **ONE** of the two blocks below

Both blocks run every stage `s0_ingest → … → report` on the **`balanced`** profile at the **same 5 cm
TSDF resolution** (`dense.tsdf_voxel_m=0.05`) and both use **exhaustive feature matching**
(`poses.matcher=exhaustive`). Exhaustive matching is essential here: the sample datasets are
**lawnmower-grid aerial surveys**, and the default *sequential* matcher only links consecutive frames,
so adjacent strips never loop-close — the trajectory drifts (~26 m on aukerman) and the mesh smears into
floating blocks. Verified locally on aukerman: trajectory RMSE **26.3 m → 6.5 m**, sparse points
**15.9k → 34.9k**, **75/75** frames registered.

They differ in **one** knob, `dense.depth_trunc_m` — the **maximum depth** the dense stage integrates.
This is not a free RAM dial: Open3D discards every pixel deeper than it, so `depth_trunc_m` **must exceed
your flying height (AGL)** or the ground is deleted.

| Block | Use on | `depth_trunc_m` | For |
|-------|--------|-----------------|-----|
| **6A · free tier** | Colab/Kaggle free T4 (**~13 GB RAM**) | **40 m** | **LOW-altitude** passes only — ground within ~40 m of the camera. |
| **6B · high-RAM (recommended)** | **Kaggle (free, ~30 GB)** or Colab **High-RAM** | **150 m** | **Aerial mapping** like the **aukerman sample** (~100–130 m AGL). Reaches the ground ~100 m below. |

> **Why aukerman needs 6B.** The aukerman flight is ~100–130 m above ground, so each camera sees the
> ground at ~100 m depth. 6A's 40 m clamp deletes it (→ the floating-fragment mesh you saw; DSM coverage
> 0.3 %). The honest fix is more RAM, **not** a coarser voxel — resolution stays 5 cm. Kaggle's free tier
> (~30 GB) holds the 150 m range at 5 cm; Colab's 13 GB free tier does not.

> **Honest caveats.**
> - Sizing `depth_trunc_m`: use ≈ **1.3 × your max AGL**. Too low deletes ground; far higher just
>   integrates unreliable long-range monocular depth and costs RAM.
> - If **6B dies with `[exit -9]`**, the runtime ran out of RAM for 150 m at 5 cm — use a bigger runtime
>   (Kaggle ~30 GB / Colab High-RAM). I will **not** coarsen the voxel to force a fit — that *is* a
>   resolution loss.
> - Why not `--profile max`? `max` selects `dense.method=gaussian` + `mesh.method=2dgs`, which this build
>   does not implement — those stages **raise loudly**. `balanced` is the real high-quality path here.

`DRISHTI_LOG_LEVEL=DEBUG` gives verbose logs, and the **entire console is tee'd** into
`logs/run_console.log`. The first run also fetches the RT-DETR and Depth-Anything-V2 (base) checkpoints
(both Apache-2.0) into the model cache.

In [ ]:
# == RUN · Block 6A — FREE TIER (~13 GB RAM):  balanced + near-field depth clamp ================
# 5 cm voxel + exhaustive matching, same as 6B, but clamps dense.depth_trunc_m=40 so s7_dense's
# host-RAM TSDF fits ~13 GB. That clamp only reconstructs ground within ~40 m of the camera -- correct
# for LOW, close passes but WRONG for high-altitude aerial grids like the aukerman sample (~100-130 m
# AGL): the ground is ~100 m away, so 40 m deletes it and you get floating fragments. aukerman -> use 6B.
# >>> Run 6A OR 6B, not both. Each block is self-contained. <<<
import os, subprocess, sys, time
from pathlib import Path

os.environ["DRISHTI_LOG_LEVEL"] = "DEBUG"                 # verbose, honest logs
os.environ["DRISHTI_RUNS_DIR"] = str(REPO_DIR / "runs")   # where inspect/verify/server look, too

FREE_DEPTH_TRUNC_M = 40   # near-field clamp; MUST exceed your flying height (AGL) or it deletes the ground

def _run_pipeline(extra_set=None, tag="run"):
    """Tee'd pipeline runner. Sets the global BUNDLE for §7+. Shared shape across 6A/6B."""
    global BUNDLE
    if RESUME_BUNDLE:
        bundle_path = Path(RESUME_BUNDLE)
        cmd = ["drishti", "resume", str(bundle_path)]   # resume reuses the recorded config (see 6B note)
    else:
        assert MISSION and Path(MISSION).is_file(), "no MISSION — run an import cell in §5 first."
        run_id = time.strftime(f"colab-{tag}-%Y%m%d-%H%M%S")
        bundle_path = REPO_DIR / "runs" / run_id
        (bundle_path / "logs").mkdir(parents=True, exist_ok=True)   # so we can tee before the run starts
        cmd = ["drishti", "run", "--dataset", MISSION, "--profile", "balanced", "--run-id", run_id]
        for kv in (extra_set or []):
            cmd += ["--set", kv]

    logfile = bundle_path / "logs" / "run_console.log"
    print(">>", " ".join(cmd))
    print("   console tee ->", logfile, "\n")

    with open(logfile, "w", encoding="utf-8") as lf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            sys.stdout.write(line)
            lf.write(line)
        proc.wait()

    BUNDLE = str(bundle_path)
    print(f"\n[exit {proc.returncode}]  bundle: {BUNDLE}")
    if proc.returncode == 0:
        return
    if proc.returncode in (-9, 137):   # SIGKILL — almost always the Linux OOM killer on the free tier
        print("\n[exit -9] = the OS OOM-killed the run (host RAM exhausted), NOT a DRISHTI error.\n"
              "  • Lower FREE_DEPTH_TRUNC_M (e.g. 32, 24) and re-run 6A, or\n"
              "  • switch to a High-RAM runtime and run 6B (Runtime → Change runtime type → High-RAM).\n"
              "  Completed stages are cached — set  RESUME_BUNDLE = BUNDLE  to continue where it died.")
    else:
        print("A stage failed above — that is the honest, fail-loud behaviour. Inspect it in §7, fix the "
              "cause, then set  RESUME_BUNDLE = BUNDLE  and re-run; completed stages are skipped.")

# LOW-ALTITUDE free-tier run: 5 cm voxel + exhaustive matching (cross-strip loop closure; sequential
# drifts ~26 m and smears the mesh -- verified locally on aukerman: RMSE 26.3->6.5 m, sparse 15.9k->34.9k,
# 75/75 frames). The clamp below only reaches ground within FREE_DEPTH_TRUNC_M of the camera; for the
# ~100 m AGL aukerman sample that deletes the ground -- run 6B (Kaggle 30 GB / High-RAM) instead.
_run_pipeline(extra_set=["poses.matcher=exhaustive", f"dense.depth_trunc_m={FREE_DEPTH_TRUNC_M}"], tag="free")

In [ ]:
# == RUN · Block 6B — HIGH-RAM (>= 30 GB): full quality + exhaustive matching (RECOMMENDED) =====
# Run this on Kaggle (free, ~30 GB RAM) or a Colab High-RAM runtime. This is the RIGHT block for the
# aukerman sample and any aerial-mapping flight: (1) exhaustive matching so the lawnmower strips loop-
# close (sequential drifts ~26 m -> smeared mesh), and (2) dense.depth_trunc_m=150 so the dense stage
# reaches the GROUND ~100 m below the ~100-130 m AGL cameras (a lower clamp deletes it -> floating
# blocks). 5 cm voxel unchanged -- full resolution. >>> Run 6A OR 6B, not both; 6B does not need 6A. <<<
import os, subprocess, sys, time
from pathlib import Path

os.environ["DRISHTI_LOG_LEVEL"] = "DEBUG"
os.environ["DRISHTI_RUNS_DIR"] = str(REPO_DIR / "runs")

# NOTE on resume: `drishti resume` reuses the config recorded in the bundle's manifest, so resuming a
# 6A bundle keeps 6A's near-field clamp. To get 6B's ground-reaching 150 m range + exhaustive matching,
# start a FRESH run (RESUME_BUNDLE = None in §5) rather than resuming a 6A bundle.
if RESUME_BUNDLE:
    bundle_path = Path(RESUME_BUNDLE)
    cmd = ["drishti", "resume", str(bundle_path)]
else:
    assert MISSION and Path(MISSION).is_file(), "no MISSION — run an import cell in §5 first."
    run_id = time.strftime("colab-full-%Y%m%d-%H%M%S")
    bundle_path = REPO_DIR / "runs" / run_id
    (bundle_path / "logs").mkdir(parents=True, exist_ok=True)
    # exhaustive matching (cross-strip loop closure) + depth_trunc=150 (reach the ground ~100 m below
    # the ~100-130 m AGL flight); 5 cm voxel unchanged. Both are required -- see the header.
    cmd = ["drishti", "run", "--dataset", MISSION, "--profile", "balanced", "--run-id", run_id,
           "--set", "poses.matcher=exhaustive", "--set", "dense.depth_trunc_m=150"]

logfile = bundle_path / "logs" / "run_console.log"
print(">>", " ".join(cmd))
print("   console tee ->", logfile, "\n")

with open(logfile, "w", encoding="utf-8") as lf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        lf.write(line)
    proc.wait()

BUNDLE = str(bundle_path)
print(f"\n[exit {proc.returncode}]  bundle: {BUNDLE}")
if proc.returncode in (-9, 137):
    print("\n[exit -9] = the OS OOM-killed the run — even this runtime ran out of RAM for the 150 m\n"
          "  range at 5 cm. Use a bigger runtime (Kaggle ~30 GB / Colab High-RAM). Do NOT fall back to 6A for"
          " this aerial dataset -- 6A's 40 m clamp deletes the ground.")
elif proc.returncode != 0:
    print("A stage failed above — that is the honest, fail-loud behaviour. Inspect it in §7, fix the "
          "cause, then set  RESUME_BUNDLE = BUNDLE  and re-run; completed stages are skipped.")

## 7 · Inspect + verify (the ground truth)

`inspect` prints the manifest — per-stage status, where it ran, wall-time, **measured** confidence, and
any degradation. `verify` recomputes every output hash and checks it against the manifest.

In [ ]:
!drishti inspect "$BUNDLE"
!drishti verify "$BUNDLE"

## 8 · Logs & report

Everything the run recorded: the tee'd console log, the per-stage manifest (above), and the accuracy
report the final stage wrote.

In [ ]:
import json
from pathlib import Path
B = Path(BUNDLE)

print("=== logs/ ===")
for p in sorted((B / "logs").glob("*")):
    print(f"  {p.name:24} {p.stat().st_size:>10,} B")

rc = B / "logs" / "run_console.log"
if rc.is_file():
    print("\n=== last 25 lines of run_console.log ===")
    print("".join(rc.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-25:]))

print("=== report/ ===")
for p in sorted((B / "report").glob("*")):
    print("  ", p.name)

rj = B / "report" / "report.json"
if rj.is_file():
    r = json.loads(rj.read_text(encoding="utf-8"))
    print("\n=== report.json (summary) ===")
    for k in ("drishti_version", "total_wall_seconds", "n_failed", "n_degraded"):
        if k in r:
            print(f"  {k}: {r[k]}")
    if "accuracy" in r:
        print("  accuracy:", json.dumps(r["accuracy"]))
    print("\nOpen report/report.html locally for the full formatted report.")

## 9 · Export EVERYTHING — download the whole bundle (incl. logs)

Two archives:
- **full** — the complete bundle: `manifest.json`, `inputs/`, `logs/`, `report/`, and every stage dir
  (raw frames, depth, dense cloud, mesh, and all `s10_export` deliverables). Everything, reproducible.
- **slim** — just deliverables + report + logs + manifest (no raw inputs/intermediates), for a quick share.

Big real sets (aukerman/lewis) make the full zip large; if a browser download stalls, copy it to Google
Drive instead (commented below).

In [ ]:
import shutil, zipfile
from pathlib import Path
B = Path(BUNDLE)
rid = B.name

# --- full bundle ---
full_zip = shutil.make_archive(str(B.parent / f"{rid}_full"), "zip", root_dir=B.parent, base_dir=rid)
print("full :", full_zip, f"({Path(full_zip).stat().st_size / 1e6:.1f} MB)")

# --- slim: deliverables + report + logs + manifest ---
slim_zip = str(B.parent / f"{rid}_slim.zip")
with zipfile.ZipFile(slim_zip, "w", zipfile.ZIP_DEFLATED) as z:
    if (B / "manifest.json").is_file():
        z.write(B / "manifest.json", f"{rid}/manifest.json")
    for sub in ["report", "logs", "s10_export"]:
        for p in (B / sub).rglob("*"):
            if p.is_file():
                z.write(p, f"{rid}/{p.relative_to(B)}")
print("slim :", slim_zip, f"({Path(slim_zip).stat().st_size / 1e6:.1f} MB)")

# --- download (Colab) ---
try:
    from google.colab import files
    files.download(slim_zip)      # small first; swap to full_zip for everything
    # files.download(full_zip)
except Exception:
    print("Not on Colab — grab the archives from the file browser (Kaggle: /kaggle/working).")

# --- or copy to Drive for large bundles ---
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy2(full_zip, "/content/drive/MyDrive/")

## 10 · Round-trip back to your machine

You ran the whole pipeline here, so locally you only need to **view / serve** it — no heavy deps required:

```bash
unzip <run_id>_full.zip -d runs/
drishti inspect runs/<run_id>            # the same manifest you saw above
drishti verify  runs/<run_id>            # confirm the download is intact

pip install -e ".[server]"
python -m server.app                     # http://localhost:8000/api/health
cd viewer && npm install && npm run dev   # browse the georeferenced model
```

`drishti resume runs/<run_id>` is a no-op once every stage is `done` (it just re-checks freshness). The
`report/report.html` in the bundle is the shareable accuracy report; `s10_export/` holds the
OBJ · PLY · LAS · GeoTIFF · glTF · FBX deliverables.